In [2]:
%%writefile src/prep.py
#create data prep
import argparse
import pandas as pd 
import os 

parser=argparse.ArgumentParser()
parser.add_argument("--input_data", type=str)
parser.add_argument("--output_data", type=str)
args=parser.parse_args()

df=pd.read_csv(os.path.join(args.input_data,"diabetes.csv"))
df=df.dropna()

os.makedirs(args.output_data, exist_ok=True)
df.to_csv(os.path.join(args.output_data,'cleaned.csv'), index=False)
print("Data Prep Complete")

Overwriting src/prep.py


In [4]:
%%writefile src/training_script.py
import argparse
import pandas as pd
import os
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

parser = argparse.ArgumentParser()

parser.add_argument("--training_data", type=str)
parser.add_argument("--model_output", type=str)

args = parser.parse_args()

df = pd.read_csv(os.path.join(args.training_data, "cleaned.csv"))

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

os.makedirs(args.model_output, exist_ok=True)

with open(os.path.join(args.model_output, "model.pkl"), "wb") as f:
    pickle.dump(model, f)

print("Training completed")


Writing src/training_script.py


In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
ws_config={
    "subscription_id": "7b1b43ca-4b64-43cf-9446-edb35a04d7d1",
    "resource_group": "AMLWS06",
    "workspace_name": "azuremlwest"
}

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(
    credential=DefaultAzureCredential()
)

Found the config file in: .\config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [4]:
from azure.ai.ml import command, Input
pipeline_input=Input(type="uri_folder",
                     path="azureml:diabetes_mltable:4")




In [5]:
compute_target = ml_client.compute.get("satyakebakshi951")


In [6]:
#Prep Step
prep_job=command(
    code="./src",
    command="python prep.py --input_data ${{inputs.input_data}} --output_data ${{outputs.cleaned_data}}",
    inputs={
        "input_data": pipeline_input
    },
    outputs={
        "cleaned_data":{"type":"uri_folder"}
    },
    environment='custom-env-scikit-train:1',
    compute='satyakebakshi951'
)

In [16]:
#Train step
train_job=command(
    code="./src",
    command="python training_script.py --training_data ${{inputs.training_data}} --model_output ${{outputs.model_output}}",
    inputs={
        "training_data": Input(type="uri_folder")
    },
    outputs={
        "model_output":{"type":"uri_folder"}
    },
    environment='custom-env-scikit-train:1',
    compute='satyakebakshi951'
)

In [17]:
from azure.ai.ml.dsl import pipeline
@pipeline()
def prep_train_pipeline(pipeline_input):
    prep=prep_job()
    prep.inputs.input_data=pipeline_input
    train=train_job()
    train.inputs.training_data=prep.outputs.cleaned_data
    return {
        "model_output": train.outputs.model_output
    }

In [18]:
pipeline_job=prep_train_pipeline(pipeline_input=pipeline_input)

In [19]:
ml_client.jobs.create_or_update(pipeline_job)

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


Experiment,Name,Type,Status,Details Page
AzureDP100,happy_wheel_p2nk61v2gb,pipeline,NotStarted,Link to Azure Machine Learning studio


In [ ]:
#saving components
prep_save=ml_client.components.create_or_update(prep_job)
train_save=ml_client.components.create_or_update(train_job)